# Car Price Predictive Analysis

# Notebook 1: Data Preparation and Cleaning

## Purpose

This notebook prepares the original car price dataset for exploratory data analysis, statistical testing, and machine-learning modelling. It focuses on assessing data quality, correcting inconsistencies, investigating potential outliers, and creating useful features for further analysis.

## Objectives

- Load and inspect the original dataset.
- Examine data types, missing values, and duplicate records.
- Identify and investigate potential outliers in relevant numerical features.
- Standardise car brand names and correct identified data-quality issues.
- Create derived features, including `avg_mpg` and `power_to_weight`.
- Validate the cleaned dataset and check its consistency.
- Export the cleaned dataset for use in subsequent notebooks.

## Methodology

The dataset will be inspected systematically before cleaning. Data-quality issues will be addressed using appropriate methods, while potential outliers will be investigated rather than automatically removed, as extreme values may represent genuine vehicle characteristics.

Feature engineering will be used to create additional variables that may support the analysis. The final dataset will be validated before being exported.

## Expected Output

The cleaned dataset will be saved as `outputs/car_prices_cleaned.csv`. It will serve as the input for:

- **Notebook 2:** Exploratory Data Analysis and Statistics.
- **Notebook 3:** Machine Learning and Evaluation.

---

## Change Working Directory

The notebook is stored inside the `jupyter_notebooks` folder, while the project files are stored in the parent project folder.

The working directory is changed to the project folder so that the dataset and other project files can be accessed using relative file paths.

In [1]:
import os
project_dir = r"D:\car-price-predictive-analysis"

os.chdir(project_dir)
print(os.getcwd())  

D:\car-price-predictive-analysis


## Import Libraries

This section imports the Python libraries required for data manipulation, numerical analysis and data visualisation.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ModuleNotFoundError: No module named 'seaborn'

## Load the Dataset

This section loads the car-price dataset into a pandas DataFrame, ready for initial inspection and subsequent data analysis.

In [ ]:
# Load the car price dataset

df = pd.read_csv('inputs/CarPrice_Assignment.csv')

df.head()

In [ ]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nDoes the CSV exist at the expected path?")
print(Path("inputs/CarPrice_Assignment.csv").exists())

print("\nCSV files found in the project folders:")
for file in Path.cwd().rglob("CarPrice_Assignment.csv"):
    print(file) 

## Initial Dataset Inspection

This section examines the dataset's dimensions and column data types to understand its structure before carrying out data quality checks and exploratory data analysis.


In [ ]:
# Check the dataset dimensions and column data types

print("Dataset shape:", df.shape)

print("\nColumn names and data types:")
df.info()

---

> **Key Findings**
>
> - The dataset contains **205 rows and 26 columns**.
> - All 26 columns report 205 non-null entries, so the initial inspection shows no missing values.
> - The columns have three data types: `float64` (8 columns), `int64` (8 columns), and `object` (10 columns).
> - The `price` column is numeric and will be the target variable for the regression model.
>
> **Note:** Duplicate records and other potential data-quality issues have not yet been assessed.

## Missing Value Assessment

This check identifies the number of missing values in each column to determine whether any missing-data treatment is required before further analysis.

In [ ]:
df.isnull().sum()

---

> **Key Findings**
>
> - All 26 columns contain **zero missing values**.
> - No missing-value imputation or removal is required at this stage.
> - Further data-quality checks are still needed to identify duplicate records and assess potential outliers.

## Duplicate Record Assessment

This check identifies exact duplicate rows in the dataset to determine whether duplicate records may affect the reliability of the analysis.

In [ ]:
df.duplicated().sum()

---

> **Key Findings — Duplicate Records**
>
> - The dataset contains **0 exact duplicate rows**.
> - No duplicate-row removal is required based on this check.
> - This check does not identify inconsistencies or repeated values within individual columns.
>
> **Note:** All 205 records are retained at this stage. Further data-quality checks are needed to assess other potential issues.

## Outlier Detection Using the IQR Method

The Interquartile Range (IQR) method is used to identify potential outliers in numerical columns. It calculates the first quartile (Q1), third quartile (Q3), and IQR (Q3 − Q1). Values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR are flagged for further investigation.

In [ ]:
# Select numerical columns, excluding the car identifier
numeric_cols = df.select_dtypes(include='number').columns.drop(
    'car_ID', errors='ignore'
)

Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)

IQR = Q3 - Q1

outliers = ((df[numeric_cols] < (Q1 - 1.5 * IQR)) |
            (df[numeric_cols] > (Q3 + 1.5 * IQR))).sum()

outliers

---

> **Key Findings — Outlier Detection**
>
> - The IQR method is used to identify potential outliers in numerical columns.
> - Values outside the IQR boundaries are flagged for further investigation.
> - A flagged value is not necessarily an error; it may represent a genuine characteristic of a vehicle.
>
> **Note:** Outlier detection results should be reviewed before deciding whether to retain, transform, or remove any values. Identifier columns, such as `car_ID`, should not be treated as meaningful numerical measurements.

## Data Cleaning and Feature Engineering

###  Manufacturer Name Standardisation

This step examines the manufacturer names extracted from the car name column and counts their occurrences. The results will help identify spelling inconsistencies that need to be standardised before further analysis.

In [ ]:
# Extract raw brand names from CarName and assign to a new column in df
df['brand'] = df['CarName'].apply(lambda x: x.split(' ')[0].lower())

# Display sorted unique raw brand names and their frequencies
raw_brand_counts = df['brand'].value_counts().sort_index()
print(f"Total Unique Raw Brands Found: {len(raw_brand_counts)}\n")
print(raw_brand_counts)

> **Key Findings — Raw Manufacturer Name Inspection**
>
> - **Unique Extracted Brands:** Initial extraction identifies 27 unique raw manufacturer tokens across the dataset.
> - **Identified Spelling Inconsistencies and Typos:**
>   - `alfa-romero` instead of `alfa-romeo`
>   - `maxda` instead of `mazda`
>   - `porcshce` instead of `porsche`
>   - `toyouta` instead of `toyota`
>   - `vokswagen` and `vw` instead of `volkswagen`
>
> **Note:** These manufacturer name errors must be standardised to prevent duplicate brand entries during analysis and feature engineering.

###  Verify Standardised Manufacturer Names

This step checks the frequency of each standardised manufacturer name after applying the spelling corrections. Reviewing the counts helps confirm that known spelling variations have been consolidated and that the dataset is ready for further analysis.

In [ ]:
# Define manufacturer spelling corrections and aliases
brand_corrections = {
    "alfa-romero": "alfa-romeo",
    "maxda": "mazda",
    "porcshz": "porsche",
    "porcshce": "porsche",
    "toyouta": "toyota",
    "vokswagen": "volkswagen",
    "vw": "volkswagen"
}

# Apply spelling corrections to the extracted manufacturer names
df["brand"] = df["brand"].replace(brand_corrections)

# Verify the corrected manufacturer names
print("Unique cleaned brands:")
print(sorted(df["brand"].unique()))

---

> **Key Findings — Manufacturer Name Validation**
>
> - The manufacturer labels have been standardised using the defined spelling-correction dictionary.
> - The frequency counts show the number of records associated with each standardised manufacturer.
> - Reviewing these counts helps identify whether any known spelling variations remain.
>
> **Note:** The original `CarName` column has been preserved. The frequency output should be reviewed to confirm that the corrections have been applied successfully.

###  Extract Car Model Names

The `CarName` column contains both the manufacturer name and the car model. This step extracts the model name by removing the first word, which represents the manufacturer, and converting the remaining text to lowercase.

The extracted model names are stored in a new `car_model` column, while the original `CarName` column is preserved. This separation supports further exploratory data analysis and comparisons between vehicle models.

In [ ]:
# Extract the car model name from CarName
df["car_model"] = (
    df["CarName"]
    .str.split()
    .str[1:]
    .str.join(" ")
    .str.lower()
)

# Display the original name, brand, and extracted model
display(df[["CarName", "brand", "car_model"]].head(10))

---

> **Key Findings — Car Model Extraction**
>
> - The `car_model` column has been created by extracting the words following the manufacturer name in `CarName`.
> - The extracted model names have been converted to lowercase for consistency.
> - The sample output confirms that manufacturer names and model names are stored in separate columns.
> - The original `CarName` column has been preserved for reference and validation.
>
> **Note:** This step separates the existing information; it does not create new vehicle specifications or guarantee that every model name is unique.

### Standardise Car Model Names

Some car model labels may contain minor inconsistencies in spacing or naming. This step standardises selected model names using a dictionary of known corrections to improve consistency during exploratory data analysis.

In [ ]:
# Define known car model name corrections
model_corrections = {
    "100 ls": "100ls",
    "5000s (diesel)": "5000s"
}

# Apply the corrections to the car_model column
df["car_model"] = df["car_model"].replace(model_corrections)

# Verify the corrected model names
print("Updated car model examples:")
display(df[["CarName", "brand", "car_model"]].head(10))

---

> **Key Findings — Car Model Standardisation**
>
> - Selected model labels have been standardised using a predefined correction dictionary.
> - The corrections address differences in spacing and remove the diesel label from the selected model name.
> - The original `CarName` column remains unchanged, allowing the cleaned values to be checked against the source data.
>
> **Note:** These corrections are limited to the explicitly listed labels. Other model variations should be investigated before applying additional changes.

### Create Derived Business Features

Feature engineering involves creating new variables from existing data to support the business objectives of this project.

This step introduces three derived features:

- `price_per_hp`: vehicle price divided by horsepower, representing price relative to engine power.
- `power_to_weight`: horsepower divided by curb weight, representing engine power relative to vehicle weight.
- `avg_mpg`: the average of city and highway miles per gallon, providing a simple measure of fuel economy.

These features will be examined during exploratory data analysis to investigate their relationships with car prices and other vehicle characteristics.

In [ ]:
# 1. Calculate price per horsepower
df["price_per_hp"] = df["price"] / df["horsepower"]

# 2. Calculate the horsepower-to-weight ratio
df["power_to_weight"] = df["horsepower"] / df["curbweight"]

# 3. Calculate average fuel economy from city and highway MPG
df["avg_mpg"] = (df["citympg"] + df["highwaympg"]) / 2

In [ ]:
# Display the original variables and the newly created features
display(
    df[
        [
            "price",
            "horsepower",
            "curbweight",
            "citympg",
            "highwaympg",
            "price_per_hp",
            "power_to_weight",
            "avg_mpg"
        ]
    ].head(10)
)

---

> **Key Findings — Feature Engineering**
>
> - Three derived features have been created: `price_per_hp`, `power_to_weight`, and `avg_mpg`.
> - `price_per_hp` represents vehicle price relative to horsepower.
> - `power_to_weight` represents horsepower relative to curb weight.
> - `avg_mpg` is calculated as the arithmetic mean of city and highway miles per gallon.
>
> **Note:** These features provide additional variables for exploratory data analysis. Their relationships with car prices will be investigated in subsequent sections. The calculated values should be checked for unexpected or invalid results before modelling.

###  Validate Derived Features

This step checks the newly created features to confirm that the calculations have produced valid numerical values. Although the original dataset contains no missing values, validation is necessary to ensure that feature engineering has not introduced missing or infinite values through invalid mathematical operations.

In [ ]:
# Check missing values in the derived features
derived_features = [
    "price_per_hp",
    "power_to_weight",
    "avg_mpg"
]

print("Missing values:")
print(df[derived_features].isnull().sum())

print("\nInfinite values:")
print(
    df[derived_features]
    .isin([float("inf"), float("-inf")])
    .sum()
)

---

> **Key Findings — Derived Feature Validation**
>
> - `price_per_hp` contains 0 missing values and 0 infinite values.
> - `power_to_weight` contains 0 missing values and 0 infinite values.
> - `avg_mpg` contains 0 missing values and 0 infinite values.
> - The validation confirms that the three derived features contain no missing or infinite values.
>
> **Conclusion:** No additional missing-value treatment is required for these derived features at this stage.

###  Inspect the Transformed Dataset Structure

After completing the initial data cleaning and feature engineering steps, the dataset is inspected to assess its updated structure.

This check examines the number of rows and columns, reviews the column names, and displays a sample of records. It helps confirm that the original observations have been retained and that the newly created features are present before the next stage of the ETL process.

In [ ]:
# Check the dataset dimensions after feature engineering
print("Dataset shape:", df.shape)

# Display the column names
print("\nDataset columns:")
print(df.columns.tolist())

# Inspect the first five rows
display(df.head())

> **Key Findings — Transformed Dataset Structure**
>
> - The dataset contains **205 rows and 31 columns** after the transformations completed so far.
> - All 205 original records have been retained.
> - The dataset contains the original vehicle attributes and the new `brand` and `car_model` columns.
> - Three derived features have been added: `price_per_hp`, `power_to_weight`, and `avg_mpg`.
>
> **Next step:** Verify the final dataset structure and prepare the transformed dataset for export.

#### Remove Duplicate Manufacturer Column

The `Brand` and `brand` columns contain the same type of information. This step removes the redundant `Brand` column and retains the lowercase `brand` column for consistency.

In [ ]:
# Remove the duplicate Brand column, keeping the lowercase brand column
if "Brand" in df.columns:
    df = df.drop(columns=["Brand"])

# Verify the remaining brand-related columns
print("Brand-related columns:")
print([col for col in df.columns if "brand" in col.lower()])

# Check the updated dataset dimensions
print("\nUpdated dataset shape:", df.shape)

###  Verify the Final Transformed Dataset

After cleaning the manufacturer names, extracting car models, creating derived features, and removing the duplicate manufacturer column, the dataset structure is checked again.

This verification confirms the final number of rows and columns, reviews the column names, and checks that the original `CarName` column and the new `brand` and `car_model` columns are retained. It also helps confirm that the dataset is organised correctly before exporting it as a separate cleaned CSV file.

In [ ]:
# Verify the dataset dimensions
print("Final dataset shape:", df.shape)

# Display all column names
print("\nFinal column names:")
print(df.columns.tolist())

# Confirm that the original and cleaned name columns are retained
print("\nName-related columns:")
print(df[["CarName", "brand", "car_model"]].head())

---

> **Key Findings — Final Dataset Structure**
>
> - The transformed dataset contains **205 rows and 31 columns**.
> - All 205 original records have been retained.
> - The duplicate `Brand` column has been removed, leaving one standardised manufacturer column, `brand`.
> - The original `CarName` column and the extracted `car_model` column are retained.
> - The derived features `price_per_hp`, `power_to_weight`, and `avg_mpg` are included in the transformed dataset.
>
> **Next step:** Export the transformed dataset to a separate CSV file while preserving the original dataset.

## ETL: Load the Cleaned Dataset

### Export the Cleaned Dataset

In [ ]:
# Export the transformed dataset to the outputs folder
df.to_csv("outputs/car_prices_cleaned.csv", index=False)

# Confirm the export
print("Cleaned dataset exported successfully.")
print("Dataset shape:", df.shape)

---

> **Key Findings — Cleaned Dataset Export**
>
> - The transformed dataset was successfully exported to `outputs/car_prices_cleaned.csv`.
> - The exported dataset contains **205 rows and 31 columns**.
> - The CSV file preserves the cleaned manufacturer names, extracted car models, and derived business features.
> - The original dataset remains unchanged.
>
> **Conclusion:** The transformed dataset is now available as a separate file for subsequent exploratory data analysis, statistical analysis, and machine learning.